# LLM Automation for Claims Analysis

Exploring how LLMs can automate tasks in a data analytics workflow that are currently done manually:

1. **Executive summary generation** — turn metrics into a plain-English briefing
2. **Denial reason classification** — map free-text denial notes to a standard taxonomy
3. **Natural language → SQL** — let non-technical users ask questions in plain English
4. **Anomaly explanation** — flag data quality issues and explain likely root causes

---
**Running this notebook:**
- `DEMO_MODE = True` (default) — runs with mock responses, no API key needed
- Set `DEMO_MODE = False` and add your `OPENAI_API_KEY` to use live GPT-4

> I used AI tools to help build this — which is exactly what this notebook demonstrates. The point isn't whether AI helped write it; it's whether you understand it and can extend it.

In [1]:
import os
import json
import pandas as pd

DEMO_MODE = True  # Set False + add OPENAI_API_KEY for live GPT-4

try:
    from openai import OpenAI
    client = OpenAI(api_key=os.getenv('OPENAI_API_KEY', ''))
except ImportError:
    client    = None
    DEMO_MODE = True

def call_llm(system, user, mock=''):
    if DEMO_MODE or not client:
        print('[demo mode]')
        return mock
    r = client.chat.completions.create(
        model='gpt-4',
        messages=[{'role':'system','content':system},{'role':'user','content':user}],
        temperature=0.2, max_tokens=600
    )
    return r.choices[0].message.content.strip()

df = pd.read_csv('../data/sample_claims.csv')
print(f'Loaded {len(df)} claims')

Loaded 20 claims


## 1. Executive Summary Generation

Build metrics from the data, then ask GPT-4 to write a plain-English briefing.

In [2]:
paid    = df[df['claim_status'] == 'PAID']
metrics = {
    'total_claims':            len(df),
    'paid_claims':             int((df['claim_status']=='PAID').sum()),
    'denied_claims':           int((df['claim_status']=='DENIED').sum()),
    'denial_rate_pct':         round((df['claim_status']=='DENIED').mean()*100, 2),
    'total_billed':            round(df['billed_amount'].sum(), 2),
    'total_paid':              round(df['paid_amount'].sum(), 2),
    'inpatient_avg_paid':      round(df[df['claim_type']=='IP']['paid_amount'].mean(), 2),
    'behavioral_health_spend': round(df[df['provider_type']=='BEHAVIORAL']['paid_amount'].sum(), 2),
}
print(json.dumps(metrics, indent=2))

{
  "total_claims": 20,
  "paid_claims": 18,
  "denied_claims": 2,
  "denial_rate_pct": 10.0,
  "total_billed": 78150.0,
  "total_paid": 57720.0,
  "inpatient_avg_paid": 10360.0,
  "behavioral_health_spend": 450.0
}


In [3]:
mock_summary = """
Executive Summary

For the period analyzed, 20 claims were processed across TX, FL, and CA,
with $159,100 paid against $201,300 billed — a 79% average payment rate.

Notable findings:
- Denial rate is 10% (2 claims). One anomaly requires review: a denied claim
  showing a non-zero paid amount, suggesting a possible post-payment reversal
  that was not properly reconciled.
- Inpatient claims average $11,200 each — the primary cost driver.
- Behavioral health spend is $450 — relatively low but worth tracking.
- Payment rates are consistent across states (78–80%).

Recommended next steps:
1. Investigate the denied claim with paid_amount > 0.
2. Flag the two high-cost inpatient members for care management review.
"""

summary = call_llm(
    system='You are a healthcare data analyst. Summarize the following metrics '
           'in plain English for a non-technical program manager. Under 200 words. '
           'Be specific with numbers. Flag anything that warrants attention.',
    user=f'Metrics:\n{json.dumps(metrics, indent=2)}',
    mock=mock_summary
)
print(summary)

[demo mode]

Executive Summary

For the period analyzed, 20 claims were processed across TX, FL, and CA,
with $159,100 paid against $201,300 billed — a 79% average payment rate.

Notable findings:
- Denial rate is 10% (2 claims). One anomaly requires review: a denied claim
  showing a non-zero paid amount, suggesting a possible post-payment reversal
  that was not properly reconciled.
- Inpatient claims average $11,200 each — the primary cost driver.
- Behavioral health spend is $450 — relatively low but worth tracking.
- Payment rates are consistent across states (78–80%).

Recommended next steps:
1. Investigate the denied claim with paid_amount > 0.
2. Flag the two high-cost inpatient members for care management review.



## 2. Denial Reason Classification

Free-text denial notes → standard taxonomy. Useful for trend analysis and provider feedback.

In [4]:
denial_notes = [
    'Service not authorized prior to date of service',
    'Claim submitted beyond 90-day filing limit',
    'Member not enrolled on date of service',
    'Procedure code does not match diagnosis',
]

mock_classified = [
    {'note': 'Service not authorized prior to date of service',
     'category': 'AUTHORIZATION_MISSING', 'confidence': 0.97},
    {'note': 'Claim submitted beyond 90-day filing limit',
     'category': 'TIMELY_FILING',         'confidence': 0.99},
    {'note': 'Member not enrolled on date of service',
     'category': 'ELIGIBILITY_ISSUE',     'confidence': 0.95},
    {'note': 'Procedure code does not match diagnosis',
     'category': 'CODING_ERROR',          'confidence': 0.91},
]

result = call_llm(
    system='Classify each denial reason into one of: AUTHORIZATION_MISSING, '
           'DUPLICATE_CLAIM, ELIGIBILITY_ISSUE, CODING_ERROR, TIMELY_FILING, '
           'MEDICAL_NECESSITY, OTHER. Return JSON array: [{note, category, confidence}].',
    user=json.dumps(denial_notes),
    mock=json.dumps(mock_classified)
)

classified = json.loads(result) if not DEMO_MODE else mock_classified
pd.DataFrame(classified)

[demo mode]


,note,category,confidence
0,Service not authorized prior to date of service,AUTHORIZATION_MISSING,0.97
1,Claim submitted beyond 90-day filing limit,TIMELY_FILING,0.99
2,Member not enrolled on date of service,ELIGIBILITY_ISSUE,0.95
3,Procedure code does not match diagnosis,CODING_ERROR,0.91


## 3. Natural Language → SQL

An analyst describes what they want in plain English — the LLM writes the SQL.

In [5]:
SCHEMA = ('Table: claims(claim_id, member_id, state_code, plan_id, service_date, '
          'claim_type, diagnosis_code, procedure_code, billed_amount, paid_amount, '
          'claim_status, provider_id, provider_type, managed_care_flag). '
          'Return only the SQL query, no explanation.')

questions = {
    'Which members had more than one inpatient stay?':
        """
SELECT   member_id, COUNT(*) AS inpatient_stays
FROM     claims
WHERE    claim_type='IP' AND claim_status='PAID'
GROUP BY member_id
HAVING   COUNT(*) > 1
ORDER BY inpatient_stays DESC;""",

    'What is total behavioral health spend by state?':
        """
SELECT   state_code, ROUND(SUM(paid_amount),2) AS bh_spend
FROM     claims
WHERE    provider_type='BEHAVIORAL' AND claim_status='PAID'
GROUP BY state_code
ORDER BY bh_spend DESC;""",
}

for question, mock_sql in questions.items():
    print(f'\nQ: {question}')
    sql = call_llm(system=SCHEMA, user=question, mock=mock_sql)
    print(f'SQL:{sql}')


Q: Which members had more than one inpatient stay?
[demo mode]
SQL:
SELECT   member_id, COUNT(*) AS inpatient_stays
FROM     claims
WHERE    claim_type='IP' AND claim_status='PAID'
GROUP BY member_id
HAVING   COUNT(*) > 1
ORDER BY inpatient_stays DESC;

Q: What is total behavioral health spend by state?
[demo mode]
SQL:
SELECT   state_code, ROUND(SUM(paid_amount),2) AS bh_spend
FROM     claims
WHERE    provider_type='BEHAVIORAL' AND claim_status='PAID'
GROUP BY state_code
ORDER BY bh_spend DESC;


## 4. Anomaly Explanation

Rather than just flagging anomalies, get a plain-English explanation of likely root cause + fix.

In [6]:
anomalies = (
    df[(df['claim_status']=='DENIED') & (df['paid_amount']>0)]
    [['claim_id','paid_amount','claim_status']]
    .to_dict(orient='records')
)

if not anomalies:
    print('No anomalies in this dataset.')
else:
    mock_explanation = """
Anomaly — Denied claim with paid_amount > 0:

This typically occurs when a payment is processed and the claim is later
retroactively denied during a post-payment audit (e.g. eligibility review
or COB check). The paid amount was not subsequently voided in the system.

Recommended action: Cross-reference with remittance data to confirm whether
a reversal transaction was generated. If not, initiate a manual adjustment
to reconcile the record.
"""
    explanation = call_llm(
        system='You are a claims data quality analyst. For each anomaly, '
               'briefly explain the likely root cause and suggest a remediation. '
               'One paragraph per anomaly.',
        user=f'Anomalies:\n{json.dumps(anomalies, indent=2)}',
        mock=mock_explanation
    )
    print(explanation)

No anomalies in this dataset.
